In [2]:
# =======================
# Senderovich Level-3 + XGBoost
# =======================
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import pickle
from tqdm import tqdm
import time

# ---------- Utilities ----------
def _safe_end(x):
    # fall back to timestamp if end_time is missing
    return x["end_time"] if pd.notna(x["end_time"]) else x["timestamp"]

def build_case_index(df_log: pd.DataFrame, activity_to_int: dict):
    """
    Build per-case timelines and fast arrays for active-case queries.
    Returns:
      case_ids: np.array shape (C,)
      starts:   np.array shape (C,)
      ends:     np.array shape (C,)
      times_by_case: dict[case_id] -> np.array of event start times (sorted)
      acts_by_case:  dict[case_id] -> np.array of activity_int (sorted by the times above)
    """
    dfL = df_log[df_log["status"] != "gateway"].copy()
    if "end_time" not in dfL:
        dfL["end_time"] = np.nan
    dfL["end_time_filled"] = dfL.apply(_safe_end, axis=1)

    # map activities to ints (unknown -> 0)
    dfL["activity_int"] = dfL["activity"].map(activity_to_int).fillna(0).astype(int)

    # ensure sorting per case
    dfL = dfL.sort_values(["case_id", "timestamp"], kind="mergesort")

    grp = dfL.groupby("case_id", sort=False)
    starts = grp["timestamp"].min().astype(float)
    ends   = grp["end_time_filled"].max().astype(float)

    # build arrays per case
    times_by_case = {}
    acts_by_case  = {}
    for cid, g in grp:
        times_by_case[cid] = g["timestamp"].to_numpy(dtype=float)
        acts_by_case[cid]  = g["activity_int"].to_numpy(dtype=int)

    case_ids = starts.index.to_numpy(dtype=int)
    starts   = starts.to_numpy()
    ends     = ends.to_numpy()

    return case_ids, starts, ends, times_by_case, acts_by_case

def last_k_pattern_at(times_arr: np.ndarray, acts_arr: np.ndarray, t: float, k: int):
    """
    Return tuple of last k activities (as ints) up to time t for a single case.
    If no event at or before t, returns empty tuple ().
    """
    # index of last event with start_time <= t
    idx = np.searchsorted(times_arr, t, side="right") - 1
    if idx < 0:
        return ()
    start = max(0, idx - k + 1)
    return tuple(acts_arr[start:idx+1])

def level3_feature_counts_at_time(t: float,
                                  case_ids: np.ndarray,
                                  starts: np.ndarray,
                                  ends: np.ndarray,
                                  times_by_case: dict,
                                  acts_by_case: dict,
                                  k: int,
                                  focal_case_id: int):
    """
    Compute Level-3 dictionary { "PAT[a|b|c]": count } for all cases active at time t,
    excluding focal_case_id.
    """
    # active mask
    mask = (starts <= t) & (ends > t)
    active_ids = case_ids[mask]

    feat = Counter()
    for cid in active_ids:
        if cid == focal_case_id:
            continue
        pat = last_k_pattern_at(times_by_case[cid], acts_by_case[cid], t, k)
        if len(pat) == 0:
            continue
        key = "PAT[" + "|".join(map(str, pat)) + "]"
        feat[key] += 1
    return dict(feat)

def build_row_features(row, l3_counts: dict, vocab_size_acts: int):
    """
    Minimal intra-case + inter-case features:
      - elapsed since start (relative, from your 'timestamp_case')
      - prefix length
      - last activity (one-hot via DictVectorizer by using a string key)
      - Level-3 counts (already in dict)
    """
    x = {}
    # elapsed since start (relative time inside the case, consistent with your list_timestamps_case[-1])
    elapsed = row.get("timestamp_case", None)
    if elapsed is None or pd.isna(elapsed):
        # fallback: relative time = last item in list_timestamps_case
        lst = row.get("list_timestamps_case", [])
        elapsed = lst[-1] if len(lst) else 0.0
    x["elapsed_case"] = float(elapsed)

    # prefix length
    pref = row["prefix_int"] if isinstance(row["prefix_int"], list) else []
    x["prefix_len"] = len(pref)

    # last activity (categorical)
    last_act = pref[-1] if len(pref) else 0
    x[f"LAST_ACT={last_act}"] = 1  # one-hot via DictVectorizer

    # merge Level-3 counts
    x.update(l3_counts)
    return x

def build_dataset_level3(df_samples: pd.DataFrame,
                         case_index,
                         k_last=3,
                         top_m_patterns=2000,
                         fit_vectorizer: bool = True,
                         dict_vectorizer: DictVectorizer = None,
                         vocab_size_acts: int = None):
    """
    Creates (X, y, dv, pattern_stats) for a split.
    If fit_vectorizer=True, it also prunes to top_m_patterns by frequency on this split (use for train only).
    """
    case_ids, starts, ends, times_by_case, acts_by_case = case_index

    # First pass: build raw dict features and collect pattern frequencies (train only)
    X_dicts = []
    y = []
    pat_freq = Counter()

    for _, row in tqdm(df_samples.iterrows(), total=len(df_samples), desc="Level3 features"):
        t = float(row["prefix_time"])  # global time of the prediction point
        focal_cid = int(row["case_id"])
        l3 = level3_feature_counts_at_time(t, case_ids, starts, ends, times_by_case, acts_by_case, k_last, focal_cid)
        # track frequencies for pruning (train only)
        pat_freq.update(l3.keys())
        feats = build_row_features(row, l3, vocab_size_acts)
        X_dicts.append(feats)
        y.append(float(row["remaining_time"]))
    y = np.asarray(y, dtype=float)

    # Optional pruning to top-M patterns to keep the space compact (train only)
    keep_patterns = None
    if fit_vectorizer and top_m_patterns is not None:
        # sort by frequency then by key for determinism
        top_keys = [k for k, _ in pat_freq.most_common(top_m_patterns)]
        keep_patterns = set(top_keys)

        def _prune(d):
            if not d:
                return d
            out = {k: v for k, v in d.items() if not k.startswith("PAT[") or k in keep_patterns}
            return out

        X_dicts = [_prune(d) for d in X_dicts]

    # Fit/transform with DictVectorizer
    if fit_vectorizer:
        dv = DictVectorizer(sparse=True)
        X = dv.fit_transform(X_dicts)
    else:
        assert dict_vectorizer is not None, "Provide the fitted DictVectorizer for val/test."
        # also prune unseen patterns implicitly (dv ignores unknown keys)
        X = dict_vectorizer.transform(X_dicts)
        dv = dict_vectorizer

    return X, y, dv, pat_freq

def train_xgb_regressor(X, y, seed=None):
    """
    XGBoost regressor with sensible defaults.
    Tweak if you see over/underfit.
    """
    model = XGBRegressor(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=8,
        #subsample=0.8,
        #colsample_bytree=0.8,
        #reg_lambda=1.0,
        objective="reg:squarederror",
        #random_state=seed,
        n_jobs=0,
        tree_method="hist"
    )
    model.fit(X, y)
    return model

# ---------- Runner per scenario ----------
def run_level3_xgboost_for_scenario(scenario,
                                    activity_to_int,
                                    train_df,
                                    val_df,
                                    test_df,
                                    df_log,
                                    k_last=3,
                                    top_m_patterns=2000,
                                    save_dir="models_level3"):
    import os
    os.makedirs(save_dir, exist_ok=True)

    # Build case index once per scenario
    case_index = build_case_index(df_log[df_log["status"]!="gateway"], activity_to_int)
    vocab_size_acts = max(activity_to_int.values()) + 1

    # TRAIN
    X_tr, y_tr, dv, pat_freq = build_dataset_level3(
        train_df, case_index,
        k_last=k_last, top_m_patterns=top_m_patterns,
        fit_vectorizer=True, dict_vectorizer=None,
        vocab_size_acts=vocab_size_acts
    )
    t0 = time.time()
    model = train_xgb_regressor(X_tr, y_tr)#, seed=42)
    training_time = time.time() - t0

    # VAL
    X_va, y_va, _, _ = build_dataset_level3(
        val_df, case_index,
        k_last=k_last, top_m_patterns=None,
        fit_vectorizer=False, dict_vectorizer=dv,
        vocab_size_acts=vocab_size_acts
    )
    val_pred = model.predict(X_va)
    val_mae = mean_absolute_error(y_va, val_pred)
    print(f"[{scenario}] VAL MAE = {val_mae:.4f}")

    # TEST
    X_te, y_te, _, _ = build_dataset_level3(
        test_df, case_index,
        k_last=k_last, top_m_patterns=None,
        fit_vectorizer=False, dict_vectorizer=dv,
        vocab_size_acts=vocab_size_acts
    )
    test_pred = model.predict(X_te)
    test_mae = mean_absolute_error(y_te, test_pred)
    print(f"[{scenario}] TEST MAE = {test_mae:.4f}")

    # Save artifacts
    with open(f"{save_dir}/l3_dv_{scenario}.pkl", "wb") as f:
        pickle.dump(dv, f)
    with open(f"{save_dir}/l3_model_{scenario}.pkl", "wb") as f:
        pickle.dump(model, f)

    # Also save quick diagnostics
    pd.DataFrame({
        "y_true": y_te,
        "y_pred": test_pred
    }).to_csv(f"{save_dir}/l3_preds_{scenario}.csv", index=False)

    return {
        "scenario": scenario,
        "val_mae": val_mae,
        "test_mae": test_mae,
        "training_time_sec":training_time,
        "n_features": X_tr.shape[1],
        "top_m_patterns": top_m_patterns,
        "unique_patterns_seen_train": sum(1 for k in pat_freq if k.startswith("PAT["))
    }


# Build/clean the raw event log once per scenario
scenario = 'scenario_1_A'
file_name=f"dataset/RLRAM_l0.1_s00_{scenario}.csv"
df_log = pd.read_csv(file_name, index_col='Unnamed: 0')
# OPTIONAL: keep only meaningful rows (already done inside build_case_index)
# df_log = df_log[df_log['status']!='gateway']

activity_to_int = pickle.load( open( f"dataset/{scenario}_activity_to_int.p", "rb" ) )
train_df = pd.read_pickle(f'dataset/{scenario}_train_prefix.pkl')
val_df = pd.read_pickle(f'dataset/{scenario}_val_prefix.pkl')
test_df = pd.read_pickle(f'dataset/{scenario}_test_prefix.pkl')

# Run Level-3 + XGBoost
stats = run_level3_xgboost_for_scenario(
    scenario=scenario,
    activity_to_int=activity_to_int,           # from your pickle
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    df_log=df_log,
    k_last=3,                  # Senderovich Level-3: last w' events (use 3)
    top_m_patterns=2000,       # prune to top patterns for compactness; tweak if you like
    save_dir="models_level3"
)
print(stats)


In [2]:
# run_level3_multi_runs.py
import os
import pickle
import pandas as pd

# --- config ---
n_runs = 10
k_last = 3            # Level-3 (last w' events)
top_m_patterns = 2000 # cap Level-3 feature space
save_dir = "models_level3"
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

scenarios = [
    # "scenario_1_A",
    # "scenario_1_B_75_unique",
    "scenario_1_B_40_unique",
    # "scenario_1_B_20_unique",
    # "scenario_1_B",
    'bpi2020_2processes_massive_share'
]
gen_available = {
    # "scenario_1_A"
    # "scenario_1_B_75_unique",
    "scenario_1_B_40_unique",
    # "scenario_1_B_20_unique",
    # "scenario_1_B",
    'bpi2020_2processes_massive_share'
}

for scenario in scenarios:
    print(f"\n=== {scenario} ===")
    file_name = f"dataset/RLRAM_l0.5_s00_{scenario}.csv"

    # lookups/mappings
    activity_to_int = pickle.load(open(f"dataset/{scenario}_activity_to_int.p", "rb"))

    # raw event log for concurrency queries
    df_log = pd.read_csv(file_name, index_col="Unnamed: 0")

    # base splits
    train_df = pd.read_pickle(f"dataset/{scenario}_train_prefix.pkl")
    val_df   = pd.read_pickle(f"dataset/{scenario}_val_prefix.pkl")
    test_df  = pd.read_pickle(f"dataset/{scenario}_test_prefix.pkl")

    # --- multiple runs (base) ---
    run_rows = []
    for run in range(1, n_runs + 1):
        print(f"\nRun {run}/{n_runs} — {scenario}")
        # add run suffix so artifacts don't overwrite each other
        scenario_tag = f"{scenario}_run{run}"
        stats = run_level3_xgboost_for_scenario(
            scenario=scenario_tag,
            activity_to_int=activity_to_int,
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            df_log=df_log,
            k_last=k_last,
            top_m_patterns=top_m_patterns,
            save_dir=save_dir,
        )
        # keep a compact row for the per-scenario CSV
        run_rows.append({
            "run": run,
            "val_mae": stats["val_mae"],
            "test_mae": stats["test_mae"],
            "n_features": stats["n_features"],
            "top_m_patterns": stats["top_m_patterns"],
            "unique_patterns_seen_train": stats["unique_patterns_seen_train"],
        })

    # save one file per scenario
    pd.DataFrame(run_rows).to_csv(f"{results_dir}/LEVEL3_XGB_{scenario}.csv", index=False)
    print(f"Saved: {results_dir}/LEVEL3_XGB_{scenario}.csv")

    # --- GEN splits (if available) ---
    if scenario in gen_available:
        print(f"\n=== {scenario} (GEN) ===")
        train_df_g = pd.read_pickle(f"dataset/{scenario}_train_prefix_gen.pkl")
        val_df_g   = pd.read_pickle(f"dataset/{scenario}_val_prefix_gen.pkl")
        test_df_g  = pd.read_pickle(f"dataset/{scenario}_test_prefix_gen.pkl")

        run_rows_g = []
        for run in range(1, n_runs + 1):
            print(f"\nRun {run}/{n_runs} — {scenario} (GEN)")
            scenario_tag = f"{scenario}_GEN_run{run}"
            stats_g = run_level3_xgboost_for_scenario(
                scenario=scenario_tag,
                activity_to_int=activity_to_int,
                train_df=train_df_g,
                val_df=val_df_g,
                test_df=test_df_g,
                df_log=df_log,   # same log; GEN prefixes drawn from same simulation
                k_last=k_last,
                top_m_patterns=top_m_patterns,
                save_dir=save_dir,
            )
            run_rows_g.append({
                "run": run,
                "val_mae": stats_g["val_mae"],
                "test_mae": stats_g["test_mae"],
                "n_features": stats_g["n_features"],
                "top_m_patterns": stats_g["top_m_patterns"],
                "unique_patterns_seen_train": stats_g["unique_patterns_seen_train"],
            })

        pd.DataFrame(run_rows_g).to_csv(f"{results_dir}/LEVEL3_XGB_{scenario}_GEN.csv", index=False)
        print(f"Saved: {results_dir}/LEVEL3_XGB_{scenario}_GEN.csv")





=== scenario_1_B_40_unique ===

Run 1/10 — scenario_1_B_40_unique


Level3 features: 100%|██████████| 4085/4085 [00:01<00:00, 3139.15it/s]


[scenario_1_B_40_unique_run1] VAL MAE = 16.9667


Level3 features: 100%|██████████| 6238/6238 [00:02<00:00, 2953.35it/s]


[scenario_1_B_40_unique_run1] TEST MAE = 16.1043

Run 2/10 — scenario_1_B_40_unique


...

[bpi2020_2processes_massive_share_GEN_run9] TEST MAE = 2.4911

Run 10/10 — bpi2020_2processes_massive_share (GEN)


Level3 features: 100%|██████████| 275/275 [00:00<00:00, 7027.30it/s]


[bpi2020_2processes_massive_share_GEN_run10] VAL MAE = 2.6936


Level3 features: 100%|██████████| 30/30 [00:00<00:00, 5931.14it/s]

[bpi2020_2processes_massive_share_GEN_run10] TEST MAE = 2.4911
Saved: results/LEVEL3_XGB_bpi2020_2processes_massive_share_GEN.csv


REAL WORLD 

In [3]:
# run_level3_on_xes_variants.py
import os
import pickle
import pandas as pd
from pathlib import Path

# <<< make sure this import works >>>
# from your_module import run_level3_xgboost_for_scenario

# ---------------- helpers ----------------
def load_df_log_from_xes(xes_path: str) -> pd.DataFrame:
    """Load a XES file and return a df_log compatible with your Level-3 code."""
    from pm4py.objects.log.importer.xes import importer as xes_importer
    from pm4py.objects.conversion.log import converter as log_converter

    log = xes_importer.apply(xes_path)
    df = log_converter.apply(log, variant=log_converter.Variants.TO_DATA_FRAME)

    case_col = "case:concept:name"
    act_col  = "concept:name"
    time_col = "time:timestamp"

    # resource column if present
    res_col = None
    for c in ["org:resource", "Resource", "resource"]:
        if c in df.columns:
            res_col = c
            break

    if not all(c in df.columns for c in [case_col, act_col, time_col]):
        raise ValueError(f"Missing required XES columns in {xes_path}")

    proc_name = Path(xes_path).stem
    out = pd.DataFrame({
        "process":   proc_name,
        "case_id":   df[case_col].astype(str),
        "activity":  df[act_col].astype(str),
        # absolute seconds (float). Your Level-3 only needs an orderable time axis.
        "timestamp": pd.to_datetime(df[time_col]).astype("int64") / 1e9,
        "resource":  (df[res_col].astype(str) if res_col else "").fillna(""),
        "status":    "running",
    })
    # Optional columns your code might read but not strictly need:
    out["end_time"]  = pd.NA
    out["cycle_time"] = pd.NA

    out = out.sort_values(["case_id", "timestamp"], kind="mergesort").reset_index(drop=True)
    return out


def try_read(path):
    if not os.path.exists(path):
        return None
    return pd.read_pickle(path)

# ---------------- config ----------------
n_runs = 10
k_last = 3
top_m_patterns = 2000
save_dir = "models_level3_xes"
results_dir = "results_level3_xes"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

base_path = "dataset_xes"      # where the generated splits live
raw_xes_dir = "raw_dataset"    # where the .xes are

# choose which logs to run (file stem without .xes).
# you can also automatically grab all .xes:
logs = [p.stem for p in Path(raw_xes_dir).glob("*.xes")]

# ---------------- main loop ----------------
for log in logs:
    print(f"\n=== {log} ===")

    # vocab
    act_map_path = f"{base_path}/{log}_activity_to_int.p"
    if not os.path.exists(act_map_path):
        print(f"[WARN] Missing vocab for {log} at {act_map_path}. Skipping.")
        continue
    activity_to_int = pickle.load(open(act_map_path, "rb"))

    # df_log from XES
    xes_path = f"{raw_xes_dir}/{log}.xes"
    if not os.path.exists(xes_path):
        print(f"[WARN] Missing XES {xes_path}. Skipping.")
        continue
    df_log = load_df_log_from_xes(xes_path)

    # ---------- STD splits ----------
    train_std = try_read(f"{base_path}/{log}_STD_train_prefix.pkl")
    val_std   = try_read(f"{base_path}/{log}_STD_val_prefix.pkl")
    test_std  = try_read(f"{base_path}/{log}_STD_test_prefix.pkl")
    if any(x is None for x in [train_std, val_std, test_std]):
        print(f"[WARN] Missing STD split for {log}. Skipping log.")
        continue

    # ---------- run STD ----------
    rows_std = []
    for run in range(1, n_runs + 1):
        print(f"Run {run}/{n_runs} — {log} [STD]")
        scenario_tag = f"{log}_STD_run{run}"
        stats = run_level3_xgboost_for_scenario(
            scenario=scenario_tag,
            activity_to_int=activity_to_int,
            train_df=train_std,
            val_df=val_std,
            test_df=test_std,
            df_log=df_log,
            k_last=k_last,
            top_m_patterns=top_m_patterns,
            save_dir=save_dir,
        )
        rows_std.append({
            "run": run,
            "val_mae": stats["val_mae"],
            "test_mae": stats["test_mae"],
            "n_features": stats["n_features"],
            "top_m_patterns": stats["top_m_patterns"],
            "unique_patterns_seen_train": stats["unique_patterns_seen_train"],
        })
    pd.DataFrame(rows_std).to_csv(f"{results_dir}/LEVEL3_XGB_{log}_STD.csv", index=False)
    print(f"Saved: {results_dir}/LEVEL3_XGB_{log}_STD.csv")

    # ---------- TEST_PAR (test only) ----------
    test_par = try_read(f"{base_path}/{log}_TEST_PAR_test_prefix.pkl")
    if test_par is not None and len(test_par) > 0:
        rows_par = []
        for run in range(1, n_runs + 1):
            print(f"Run {run}/{n_runs} — {log} [TEST_PAR]")
            scenario_tag = f"{log}_TEST_PAR_run{run}"
            stats = run_level3_xgboost_for_scenario(
                scenario=scenario_tag,
                activity_to_int=activity_to_int,
                train_df=train_std,
                val_df=val_std,
                test_df=test_par,     # only test changes
                df_log=df_log,
                k_last=k_last,
                top_m_patterns=top_m_patterns,
                save_dir=save_dir,
            )
            rows_par.append({
                "run": run,
                "val_mae": stats["val_mae"],
                "test_mae": stats["test_mae"],
                "n_features": stats["n_features"],
                "top_m_patterns": stats["top_m_patterns"],
                "unique_patterns_seen_train": stats["unique_patterns_seen_train"],
            })
        pd.DataFrame(rows_par).to_csv(f"{results_dir}/LEVEL3_XGB_{log}_TEST_PAR.csv", index=False)
        print(f"Saved: {results_dir}/LEVEL3_XGB_{log}_TEST_PAR.csv")
    else:
        print(f"[INFO] No TEST_PAR for {log} (empty or missing). Skipped.")

    # ---------- TEST_NOPAR (test only) ----------
    test_nopar = try_read(f"{base_path}/{log}_TEST_NOPAR_test_prefix.pkl")
    if test_nopar is not None and len(test_nopar) > 0:
        rows_nop = []
        for run in range(1, n_runs + 1):
            print(f"Run {run}/{n_runs} — {log} [TEST_NOPAR]")
            scenario_tag = f"{log}_TEST_NOPAR_run{run}"
            stats = run_level3_xgboost_for_scenario(
                scenario=scenario_tag,
                activity_to_int=activity_to_int,
                train_df=train_std,
                val_df=val_std,
                test_df=test_nopar,   # only test changes
                df_log=df_log,
                k_last=k_last,
                top_m_patterns=top_m_patterns,
                save_dir=save_dir,
            )
            rows_nop.append({
                "run": run,
                "val_mae": stats["val_mae"],
                "test_mae": stats["test_mae"],
                "n_features": stats["n_features"],
                "top_m_patterns": stats["top_m_patterns"],
                "unique_patterns_seen_train": stats["unique_patterns_seen_train"],
            })
        pd.DataFrame(rows_nop).to_csv(f"{results_dir}/LEVEL3_XGB_{log}_TEST_NOPAR.csv", index=False)
        print(f"Saved: {results_dir}/LEVEL3_XGB_{log}_TEST_NOPAR.csv")
    else:
        print(f"[INFO] No TEST_NOPAR for {log} (empty or missing). Skipped.")

    # ---------- TEST_SAMERES_PAR (test only) ----------
    test_sameres = try_read(f"{base_path}/{log}_TEST_SAMERES_PAR_test_prefix.pkl")
    if test_sameres is not None and len(test_sameres) > 0:
        rows_srp = []
        for run in range(1, n_runs + 1):
            print(f"Run {run}/{n_runs} — {log} [TEST_SAMERES_PAR]")
            scenario_tag = f"{log}_TEST_SAMERES_PAR_run{run}"
            stats = run_level3_xgboost_for_scenario(
                scenario=scenario_tag,
                activity_to_int=activity_to_int,
                train_df=train_std,
                val_df=val_std,
                test_df=test_sameres,   # only test changes
                df_log=df_log,
                k_last=k_last,
                top_m_patterns=top_m_patterns,
                save_dir=save_dir,
            )
            rows_srp.append({
                "run": run,
                "val_mae": stats["val_mae"],
                "test_mae": stats["test_mae"],
                "n_features": stats["n_features"],
                "top_m_patterns": stats["top_m_patterns"],
                "unique_patterns_seen_train": stats["unique_patterns_seen_train"],
            })
        pd.DataFrame(rows_srp).to_csv(f"{results_dir}/LEVEL3_XGB_{log}_TEST_SAMERES_PAR.csv", index=False)
        print(f"Saved: {results_dir}/LEVEL3_XGB_{log}_TEST_SAMERES_PAR.csv")
    else:
        print(f"[INFO] No TEST_SAMERES_PAR for {log} (empty or missing). Skipped.")

    # ---------- GEN (all three splits) ----------
    train_gen = try_read(f"{base_path}/{log}_GEN_train_prefix.pkl")
    val_gen   = try_read(f"{base_path}/{log}_GEN_val_prefix.pkl")
    test_gen  = try_read(f"{base_path}/{log}_GEN_test_prefix.pkl")

    if all(x is not None for x in [train_gen, val_gen, test_gen]) and len(test_gen) > 0:
        rows_gen = []
        for run in range(1, n_runs + 1):
            print(f"Run {run}/{n_runs} — {log} [GEN]")
            scenario_tag = f"{log}_GEN_run{run}"
            stats = run_level3_xgboost_for_scenario(
                scenario=scenario_tag,
                activity_to_int=activity_to_int,
                train_df=train_gen,
                val_df=val_gen,
                test_df=test_gen,
                df_log=df_log,   # same underlying log
                k_last=k_last,
                top_m_patterns=top_m_patterns,
                save_dir=save_dir,
            )
            rows_gen.append({
                "run": run,
                "val_mae": stats["val_mae"],
                "test_mae": stats["test_mae"],
                "n_features": stats["n_features"],
                "top_m_patterns": stats["top_m_patterns"],
                "unique_patterns_seen_train": stats["unique_patterns_seen_train"],
            })
        pd.DataFrame(rows_gen).to_csv(f"{results_dir}/LEVEL3_XGB_{log}_GEN.csv", index=False)
        print(f"Saved: {results_dir}/LEVEL3_XGB_{log}_GEN.csv")
    else:
        print(f"[INFO] No GEN split for {log} (missing or empty). Skipped.")



=== BPIC15_1 ===


parsing log, completed traces :: 100%|██████████| 1199/1199 [00:08<00:00, 138.36it/s]


Run 1/10 — BPIC15_1 [STD]


Level3 features: 100%|██████████| 11735/11735 [00:01<00:00, 6356.41it/s]


[BPIC15_1_STD_run1] VAL MAE = 1262.2451


...

Level3 features: 100%|██████████| 7562/7562 [00:01<00:00, 3943.96it/s]


[BPI_Challenge_2012_GEN_run10] TEST MAE = 102.6539
Saved: results_level3_xes/LEVEL3_XGB_BPI_Challenge_2012_GEN.csv

=== BPIC20_InternationalDeclarations ===


parsing log, completed traces :: 100%|██████████| 6449/6449 [00:09<00:00, 684.25it/s] 


Run 1/10 — BPIC20_InternationalDeclarations [STD]


ValueError: invalid literal for int() with base 10: 'declaration 1002'